# 08_validate_embedding_consistency

**Objective:** Check that in-house and pre-computed PaECTER embeddings are consistent. Each national filing whose family has an EPO/USPTO sibling is re-encoded and compared to that sibling by cosine similarity (within-family similarity), benchmarked against the similarity of 10,000 random unrelated patent pairs.

**Inputs:** `patent_master.parquet`; `lens_to_familyid.parquet`; `lens_id_to_embedding.parquet`.

**Outputs:** `text_source_drift_sims.parquet`; `random_pair_baseline_sims.parquet`.

In [ ]:
# Imports
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

In [ ]:
# Configuration: paths, model, jurisdictions, device
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
DATA_DIR = PROC
MASTER       = PROC / "patent_master.parquet"
MAPPING      = PROC / "lens_to_familyid.parquet"
CONSOLIDATED = PROC / "lens_id_to_embedding.parquet"

MODEL_NAME   = "mpi-inno-comp/paecter"
N_PER_JUR    = 20
SEED         = 42

JURISDICTIONS = ["CA","AU","GB","MX","TW","KR","NL","SE","RU","EA"]

DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available()
          else "cpu")
print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")

## Candidate pool

In [ ]:
# Build national-filing candidates with an EPO/USPTO reference vector
pm = pd.read_parquet(MASTER, columns=["lens_id","jurisdiction","doc_number","kind","title_en","abstract_en"])
pm["primary_publno"] = pm["jurisdiction"].fillna("") + pm["doc_number"].fillna("") + pm["kind"].fillna("")

mapping = pd.read_parquet(MAPPING, engine="fastparquet").dropna(subset=["docdb_family_id"])
cons = pd.read_parquet(CONSOLIDATED)
cons = cons[cons["source"].isin(["EPO","USPTO"])]
ref_map = dict(zip(cons["lens_id"], cons["embedding"]))
print(f"Reference vectors available (EPO/USPTO): {len(ref_map):,}")

j = pm.merge(mapping[["lens_id","matched_publno","source"]], on="lens_id", how="inner")
j = j[j["lens_id"].isin(ref_map)]
nat = j[
    j["jurisdiction"].isin(JURISDICTIONS)
    & (j["primary_publno"] != j["matched_publno"])
    & j["title_en"].fillna("").str.len().gt(0)
    & j["abstract_en"].fillna("").str.len().gt(0)
].copy()
print(f"Candidate national-filing rows: {len(nat):,}")
print(nat["jurisdiction"].value_counts().reindex(JURISDICTIONS).to_string())

## Stratified sample

In [ ]:
# Draw a stratified sample per jurisdiction
sample = (nat.groupby("jurisdiction", group_keys=False)
             .apply(lambda g: g.sample(n=min(N_PER_JUR, len(g)), random_state=SEED))
             .reset_index(drop=True))
print(f"\nSample: {len(sample)}")
print(sample.groupby(["jurisdiction","source"]).size().to_string())

## Encode

In [ ]:
# Encode the national-filing title and abstract with PaECTER
print("\n=== Encoding ===")
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model.eval()
texts = (sample["title_en"].fillna("").str.strip()
         + " "
         + sample["abstract_en"].fillna("").str.strip()).tolist()
new_embs = model.encode(texts, batch_size=16, show_progress_bar=True,
                        convert_to_numpy=True, normalize_embeddings=True)

## Cosine similarity

In [ ]:
# Cosine similarity between each filing and its EPO/USPTO sibling
def norm(v):
    v = np.asarray(v, dtype=np.float32)
    n = np.linalg.norm(v)
    return v / n if n > 1e-8 else v

sims = []
for i, row in sample.iterrows():
    old = norm(ref_map[row["lens_id"]])
    new = norm(new_embs[i])
    sims.append(float(np.dot(old, new)))
sample["sim"] = sims

## Report and save

In [ ]:
# Report per-jurisdiction and overall within-family similarity, then save
def stats(s):
    a = np.asarray(s, dtype=np.float64)
    return pd.Series({
        "N":      len(a),
        "min":    a.min(),
        "p05":    np.percentile(a, 5),
        "median": np.median(a),
        "mean":   a.mean(),
        "max":    a.max(),
        "std":    a.std(ddof=1),
    })

print("\n=== Per-jurisdiction stats ===")
print(sample.groupby("jurisdiction")["sim"].apply(stats).unstack().reindex(JURISDICTIONS)
        .to_string(float_format=lambda x: f"{x:.4f}"))

print("\n=== Overall stats (within-family similarity) ===")
print(stats(sample["sim"]).to_string(float_format=lambda x: f"{x:.4f}"))

out = DATA_DIR / "text_source_drift_sims.parquet"
sample[["lens_id","jurisdiction","primary_publno","matched_publno","source","sim"]].to_parquet(out)
print(f"\nWrote: {out}")

## Random-pair baseline

In [ ]:
# Random-pair baseline similarity over the full embedding pool
print("\n=== Baseline: random unrelated patent pairs ===")
N_PAIRS = 10_000
rng = np.random.default_rng(SEED)

cons_all = pd.read_parquet(CONSOLIDATED)
emb_matrix = np.vstack([np.asarray(v, dtype=np.float32) for v in cons_all["embedding"]])
norms = np.linalg.norm(emb_matrix, axis=1, keepdims=True)
emb_matrix = emb_matrix / np.where(norms > 1e-8, norms, 1.0)
n_patents = len(emb_matrix)
print(f"  Pool: {n_patents:,} patents (all sources)")

idx_a = rng.integers(0, n_patents, size=N_PAIRS)
idx_b = rng.integers(0, n_patents, size=N_PAIRS)
mask = idx_a != idx_b
idx_a, idx_b = idx_a[mask], idx_b[mask]

base_sims = np.einsum("ij,ij->i", emb_matrix[idx_a], emb_matrix[idx_b])
print(f"  N pairs  = {len(base_sims):,}")
print(f"  min      = {base_sims.min():.4f}")
print(f"  p05      = {np.percentile(base_sims, 5):.4f}")
print(f"  median   = {np.median(base_sims):.4f}")
print(f"  mean     = {base_sims.mean():.4f}")
print(f"  max      = {base_sims.max():.4f}")
print(f"  std      = {base_sims.std(ddof=1):.4f}")

base_out = DATA_DIR / "random_pair_baseline_sims.parquet"
pd.DataFrame({"sim": base_sims}).to_parquet(base_out)
print(f"\nWrote: {base_out}")